In [1]:
!pip install mediapipe opencv-python


In [10]:
import cv2
import mediapipe as mp
import math
import time

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)  # external webcam

# --- Calibration storage ---
calibrated = False
ref_angle = 0
ref_head_drop = 0
calib_start_time = None

with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        posture = "CALIBRATING..."
        color = (0, 255, 255)

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            ear = lm[mp_pose.PoseLandmark.LEFT_EAR.value]
            shoulder = lm[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            hip = lm[mp_pose.PoseLandmark.LEFT_HIP.value]

            # --- Geometry ---
            angle = abs(
                math.degrees(
                    math.atan2(hip.y - shoulder.y, hip.x - shoulder.x) -
                    math.atan2(ear.y - shoulder.y, ear.x - shoulder.x)
                )
            )

            head_drop = ear.y - shoulder.y

            # --- Calibration phase ---
            if not calibrated:
                if calib_start_time is None:
                    calib_start_time = time.time()

                if time.time() - calib_start_time > 3:
                    ref_angle = angle
                    ref_head_drop = head_drop
                    calibrated = True
                    posture = "CALIBRATION DONE"
                    color = (0, 255, 0)
                else:
                    posture = "SIT STRAIGHT (CALIBRATING)"
                    color = (0, 255, 255)

            # --- Posture evaluation ---
            else:
                angle_diff = abs(angle - ref_angle)
                head_diff = abs(head_drop - ref_head_drop)

                if angle_diff < 10 and head_diff < 0.05:
                    posture = "CORRECT POSTURE"
                    color = (0, 255, 0)
                else:
                    posture = "INCORRECT POSTURE"
                    color = (0, 0, 255)

            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            # Debug values (keep for demo)
            cv2.putText(frame, f"Angle: {int(angle)}",
                        (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (255, 255, 255), 2)

        cv2.putText(frame, f"Posture: {posture}",
                    (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                    color, 2)

        cv2.imshow("Seated Posture Assessment", frame)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
